# 🌳 ICU Clinical Grouping — LightGBM Tabular Baseline

A tabular baseline that flattens the 12-step × 68-channel time series into
per-channel summary statistics (mean, min, max, last, slope, missingness),
then trains a LightGBM multi-class classifier.

**Why this notebook exists.** Tree ensembles on summary statistics are
frequently competitive with sequence models on short ICU windows. If LightGBM
matches or beats the LSTM, that means the LSTM isn't really exploiting
temporal structure — it's effectively doing a bag-of-statistics with extra
parameters. Either way, this is the baseline reviewers will ask for.

**Compatibility.** Outputs are written to the same schema as the LSTM notebook
(`debate_inputs.json`, `model_config.json`, `feature_scaler.pkl`-style
artifact, `test_predictions.csv`) so the existing agent notebook can be
pointed at this directory by changing one path.

**Same train/val/test split as LSTM** (seed=42, 70/15/15, patient-grouped) so
top-K numbers are directly comparable.

## 1. Setup & Configuration

In [ ]:
!pip install -q lightgbm scikit-learn joblib pandas

from google.colab import drive
drive.mount('/content/drive')
print('Setup complete ✓')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete ✓


In [ ]:
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
CARDS_PATH = '/content/drive/MyDrive/702 project/patient_cards_v6_grouped.json'
OUTPUT_DIR = '/content/drive/MyDrive/702 project/TOP5_lightgbm_outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Split (MUST match LSTM notebook for fair comparison) ──────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
SPLIT_SEED  = 42

# ── LightGBM hyperparameters ──────────────────────────────────────────────────
# Reasonable defaults for multiclass log-loss on tabular medical data.
# Tuning further is unlikely to move the headline by more than ±1-2%.
LGB_PARAMS = {
    'objective'        : 'multiclass',
    'metric'           : 'multi_logloss',
    'num_leaves'       : 63,        # roughly 2^depth - 1; modest tree complexity
    'learning_rate'    : 0.05,
    'feature_fraction' : 0.8,       # column subsampling per tree
    'bagging_fraction' : 0.8,       # row subsampling per iteration
    'bagging_freq'     : 5,
    'min_data_in_leaf' : 20,
    'lambda_l2'        : 1.0,       # L2 regularization
    'verbose'          : -1,
    'num_threads'      : -1,
    'seed'             : 42,
}
NUM_BOOST_ROUND        = 2000
EARLY_STOPPING_ROUNDS  = 50

# ── Output ───────────────────────────────────────────────────────────────────
TOP_K_DEBATE = 3

print(f'Output dir : {OUTPUT_DIR}')
print(f'Split seed : {SPLIT_SEED}  (must match LSTM)')

Output dir : /content/drive/MyDrive/702 project/TOP5_lightgbm_outputs
Split seed : 42  (must match LSTM)


## 2. Load Patient Cards

In [ ]:
import json
import numpy as np

print(f'Loading {CARDS_PATH} …')
with open(CARDS_PATH) as f:
    data = json.load(f)

cards         = data['patients']
label_map     = data['label_map']
feature_names = data['feature_names']
n_features    = data['n_features']
n_timesteps   = data['n_timesteps']
num_classes   = len(label_map)
inv_label_map = {v: k for k, v in label_map.items()}

# Feature columns are interleaved [val_0, mask_0, val_1, mask_1, ...]
val_idx  = list(range(0, n_features, 2))
mask_idx = list(range(1, n_features, 2))

# Raw variable names (strip the '_val' suffix from value columns)
raw_var_names = [feature_names[i].replace('_val', '') for i in val_idx]

# ── Disambiguate duplicates ───────────────────────────────────────────────────
# MIMIC sometimes records the same physiologic measurement under multiple
# itemids (e.g., sodium from chemistry vs. blood-gas panels). Those land as
# separate columns with the same display name. We keep all the data and just
# uniquify the names with __1 / __2 suffixes.
from collections import defaultdict, Counter

seen_count   = defaultdict(int)
var_names    = []
rename_log   = []

for raw in raw_var_names:
    seen_count[raw] += 1
    if seen_count[raw] == 1:
        # First occurrence — leave as-is for now (might rename in pass 2 if it dups)
        var_names.append(raw)
    else:
        suffixed = f'{raw}__{seen_count[raw]}'
        var_names.append(suffixed)
        rename_log.append((raw, suffixed))

# Pass 2: any name that ended up duplicated (was seen 2+ times) — also suffix
# its FIRST occurrence to make ordering consistent (foo, foo__2, foo__3 → foo__1, foo__2, foo__3).
final_counts = Counter(raw_var_names)
needs_first_rename = {name for name, c in final_counts.items() if c > 1}
if needs_first_rename:
    first_seen_at = {}
    for i, name in enumerate(raw_var_names):
        if name in needs_first_rename and name not in first_seen_at:
            first_seen_at[name] = i
            var_names[i] = f'{name}__1'
            rename_log.append((name, f'{name}__1'))

# Final sanity: every name unique
assert len(set(var_names)) == len(var_names), \
    f'Still duplicated: {[n for n, c in Counter(var_names).items() if c > 1]}'

n_vars = len(var_names)

if rename_log:
    print(f'\nDisambiguated {len({orig for orig, _ in rename_log})} duplicate names:')
    by_orig = defaultdict(list)
    for orig, new in rename_log:
        by_orig[orig].append(new)
    for orig, news in sorted(by_orig.items()):
        print(f'  {orig!r:<30} → {sorted(set(news))}')
else:
    print('\nNo duplicates in feature names — nothing to rename.')

# Sanity check on data shape (unchanged from before)
c0  = cards[0]
ts0 = np.array(c0['time_series'])
assert ts0.shape == (n_timesteps, n_features), f'Shape mismatch: {ts0.shape}'
assert isinstance(c0['label'], int)
assert np.isfinite(ts0).all(), 'NaN/Inf in time_series'

print(f'\nCards     : {len(cards):,}')
print(f'Classes   : {num_classes}')
print(f'Tensor    : ({n_timesteps}, {n_features}) per patient')
print(f'Variables : {n_vars} (each with value + mask channel)')

TOP_5_CLASSES = ['SEPSIS', 'STROKE_NEURO', 'ACUTE_MI',
                 'CORONARY_ARTERY_DZ', 'HEART_FAILURE']

# Filter
keep_label_ids = {label_map[c] for c in TOP_5_CLASSES}
cards = [c for c in cards if c['label'] in keep_label_ids]

# Remap labels to 0..4
old_to_new = {old: new for new, old in enumerate(sorted(keep_label_ids))}
new_label_map = {inv_label_map[old]: new for old, new in old_to_new.items()}
new_inv_label_map = {v: k for k, v in new_label_map.items()}

for c in cards:
    c['label'] = old_to_new[c['label']]

num_classes = 5
label_map = new_label_map
inv_label_map = new_inv_label_map

Loading /content/drive/MyDrive/702 project/patient_cards_v6_grouped.json …

Disambiguated 4 duplicate names:
  'anion_gap'                    → ['anion_gap__1', 'anion_gap__2']
  'chloride'                     → ['chloride__1', 'chloride__2']
  'hemoglobin'                   → ['hemoglobin__1', 'hemoglobin__2']
  'sodium'                       → ['sodium__1', 'sodium__2']

Cards     : 28,467
Classes   : 21
Tensor    : (12, 136) per patient
Variables : 68 (each with value + mask channel)


In [ ]:
from collections import Counter

# ── Step 1: compute class frequencies ─────────────────────────────────────────
label_counts = Counter(c['label'] for c in cards)

# Top-5 most common label IDs
top5_label_ids = [label for label, _ in label_counts.most_common(5)]

# Convert to label names (for readability)
top5_label_names = [inv_label_map[l] for l in top5_label_ids]

print("\nTop 5 classes by frequency:")
for i, (lid, count) in enumerate(label_counts.most_common(5)):
    print(f"  {i}: {inv_label_map[lid]} (id={lid}) → {count:,} cards")

# ── Step 2: filter cards ──────────────────────────────────────────────────────
cards = [c for c in cards if c['label'] in top5_label_ids]

# ── Step 3: remap labels to 0..4 (preserve frequency order) ───────────────────
old_to_new = {old: new for new, old in enumerate(top5_label_ids)}

new_label_map = {inv_label_map[old]: new for old, new in old_to_new.items()}
new_inv_label_map = {v: k for k, v in new_label_map.items()}

for c in cards:
    c['label'] = old_to_new[c['label']]

num_classes = 5
label_map = new_label_map
inv_label_map = new_inv_label_map

# ── Step 4: print counts AFTER filtering/remapping ────────────────────────────
new_counts = Counter(c['label'] for c in cards)

print("\nCounts after filtering + remapping:")
for new_id in range(5):
    print(f"  {new_id}: {inv_label_map[new_id]} → {new_counts[new_id]:,} cards")



Top 5 classes by frequency:
  0: SEPSIS (id=3) → 5,524 cards
  1: STROKE_NEURO (id=4) → 4,740 cards
  2: ACUTE_MI (id=0) → 2,361 cards
  3: CORONARY_ARTERY_DZ (id=1) → 2,335 cards
  4: HEART_FAILURE (id=2) → 1,794 cards

Counts after filtering + remapping:
  0: SEPSIS → 5,524 cards
  1: STROKE_NEURO → 4,740 cards
  2: ACUTE_MI → 2,361 cards
  3: CORONARY_ARTERY_DZ → 2,335 cards
  4: HEART_FAILURE → 1,794 cards


## 3. Patient-Level Split (identical to LSTM notebook)

In [ ]:
import random
from collections import defaultdict, Counter

random.seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)

# Group by patient_id so the same patient never spans train/val/test
patient_to_cards = defaultdict(list)
cards = sorted(cards, key=lambda x: x['patient_id'])

for i, card in enumerate(cards):
    patient_to_cards[card['patient_id']].append(i)

patient_ids = list(patient_to_cards.keys())
random.shuffle(patient_ids)

n_train = int(len(patient_ids) * TRAIN_RATIO)
n_val   = int(len(patient_ids) * VAL_RATIO)

# IMPORTANT: keep these as ordered lists, not sets. set iteration order is
# nondeterministic across Python sessions because of PYTHONHASHSEED, which
# bit us before. Use sets only for membership testing if needed.
train_pids = patient_ids[:n_train]
val_pids   = patient_ids[n_train:n_train + n_val]
test_pids  = patient_ids[n_train + n_val:]

train_cards = [cards[i] for pid in train_pids for i in patient_to_cards[pid]]
val_cards   = [cards[i] for pid in val_pids   for i in patient_to_cards[pid]]
test_cards  = [cards[i] for pid in test_pids  for i in patient_to_cards[pid]]

print(f'Patients — train: {len(train_pids):,}  val: {len(val_pids):,}  test: {len(test_pids):,}')
print(f'Stays    — train: {len(train_cards):,}  val: {len(val_cards):,}  test: {len(test_cards):,}')

# Sanity: print a hash of the test cohort so we can verify it matches the LSTM
import hashlib
test_signature = hashlib.md5(
    ','.join(str(c['patient_id']) + ':' + str(c.get('stay_id', '')) for c in test_cards).encode()
).hexdigest()
print(f'\nTest cohort signature: {test_signature}')
print('  (should match between LSTM and LightGBM runs to validate fair comparison)')

Patients — train: 9,624  val: 2,062  test: 2,063
Stays    — train: 11,716  val: 2,554  test: 2,484

Test cohort signature: cecc29fd0d5283d8b86fb5be8fc250c5
  (should match between LSTM and LightGBM runs to validate fair comparison)


## 4. Feature Engineering — Time Series → Tabular

For each of the 68 variables, compute summary statistics across the 12-hour
window. These are the same statistics the agent's prompt formatter shows the
LLM, so this baseline is asking: *can a tree ensemble do as well as the LSTM
using only what the agent already sees?*

Per-variable features (6 total):
- **mean** (over observed timesteps)
- **min** (over observed timesteps)
- **max** (over observed timesteps)
- **last** (most recent observed value)
- **slope** (linear trend across observed timesteps)
- **missing_rate** (fraction of timesteps with no observation)

Total feature count: `n_vars × 6`. With 68 variables that's ~408 features.

In [ ]:
def extract_features(time_series_array):
    """
    Convert (N, T, F) raw time series to (N, n_vars * 6) tabular feature matrix.

    The mask channel tells us which timesteps are real observations vs. zeros
    from imputation. We compute statistics over OBSERVED timesteps only — when
    no timesteps are observed, we emit NaN and let LightGBM handle missingness
    natively (it splits NaN as a third branch).
    """
    X = np.asarray(time_series_array, dtype=np.float32)
    N, T, F = X.shape

    vals  = X[:, :, val_idx]   # (N, T, n_vars)  raw values
    masks = X[:, :, mask_idx]  # (N, T, n_vars)  1 = observed, 0 = missing

    # 1. Stats over observed timesteps
    obs_count = masks.sum(axis=1)                                   # (N, n_vars)
    has_obs   = obs_count > 0

    # Mean of observed
    sum_vals = (vals * masks).sum(axis=1)                           # (N, n_vars)
    with np.errstate(invalid='ignore', divide='ignore'):
        mean_obs = np.where(has_obs, sum_vals / obs_count, np.nan)

    # Min/max of observed (mask out unobserved with +/- inf, then take min/max)
    vals_for_min = np.where(masks > 0, vals,  np.inf)
    vals_for_max = np.where(masks > 0, vals, -np.inf)
    min_obs = vals_for_min.min(axis=1)
    max_obs = vals_for_max.max(axis=1)
    min_obs = np.where(has_obs, min_obs, np.nan)
    max_obs = np.where(has_obs, max_obs, np.nan)

    # 2. Last observed value (walk timesteps backward, find most recent observation)
    last_obs = np.full((N, len(val_idx)), np.nan, dtype=np.float32)
    for t in range(T - 1, -1, -1):
        # For each (sample, var) where we haven't filled in last_obs yet AND mask is set, fill it.
        unfilled = np.isnan(last_obs)
        observed = masks[:, t, :] > 0
        fill_now = unfilled & observed
        last_obs[fill_now] = vals[:, t, :][fill_now]

    # 3. Slope: simple least-squares fit of value vs. timestep over observed points.
    # When fewer than 2 points are observed, slope = 0 (no trend information).
    t_arr = np.arange(T, dtype=np.float32)
    slope = np.zeros((N, len(val_idx)), dtype=np.float32)

    # Sums needed for closed-form slope (weighted by mask):
    sum_w   = masks.sum(axis=1)                          # (N, n_vars)  = obs_count
    sum_t   = (masks * t_arr[None, :, None]).sum(axis=1)
    sum_v   = (masks * vals).sum(axis=1)
    sum_tv  = (masks * vals * t_arr[None, :, None]).sum(axis=1)
    sum_tt  = (masks * (t_arr ** 2)[None, :, None]).sum(axis=1)

    denom = sum_w * sum_tt - sum_t ** 2
    enough_pts = sum_w >= 2
    valid = enough_pts & (denom != 0)

    with np.errstate(invalid='ignore', divide='ignore'):
        slope_calc = (sum_w * sum_tv - sum_t * sum_v) / denom
    slope = np.where(valid, slope_calc, 0.0)

    # 4. Missingness rate
    missing_rate = 1.0 - (obs_count / T)

    # Stack all 6 stats per variable: (N, 6, n_vars) → (N, n_vars * 6)
    stacked = np.stack([mean_obs, min_obs, max_obs, last_obs, slope, missing_rate], axis=1)
    flat = stacked.reshape(N, -1)
    return flat


# Build feature names in the same order: var0_mean, var0_min, ..., var0_missing, var1_mean, ...
STAT_NAMES = ['mean', 'min', 'max', 'last', 'slope', 'missing']
tabular_feature_names = []
for stat in STAT_NAMES:
    for v in var_names:
        tabular_feature_names.append(f'{v}__{stat}')
# Order matches the np.stack above (stat-major), so reshape gives this order
n_tab_features = len(tabular_feature_names)
print(f'Tabular features per stay: {n_tab_features}  ({n_vars} vars × {len(STAT_NAMES)} stats)')

Tabular features per stay: 408  (68 vars × 6 stats)


In [ ]:
# Materialize feature matrices.
# Loading all time_series at once may use ~1-2 GB depending on dataset size — fine on Colab.
print('Building feature matrices...')

def cards_to_array(cards_list):
    return np.array([c['time_series'] for c in cards_list], dtype=np.float32)

X_train_raw = cards_to_array(train_cards)
X_val_raw   = cards_to_array(val_cards)
X_test_raw  = cards_to_array(test_cards)

X_train = extract_features(X_train_raw)
X_val   = extract_features(X_val_raw)
X_test  = extract_features(X_test_raw)

y_train = np.array([c['label'] for c in train_cards], dtype=np.int64)
y_val   = np.array([c['label'] for c in val_cards],   dtype=np.int64)
y_test  = np.array([c['label'] for c in test_cards],  dtype=np.int64)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val  : {X_val.shape}    y_val  : {y_val.shape}')
print(f'X_test : {X_test.shape}   y_test : {y_test.shape}')

# Free raw arrays — we don't need them past this point
del X_train_raw, X_val_raw, X_test_raw
import gc; gc.collect()

# Quick sanity: NaN rate per column should be reasonable (mostly = missing rate)
nan_rate = np.isnan(X_train).mean(axis=0)
print(f'\nNaN rate per feature — mean: {nan_rate.mean():.2%}  max: {nan_rate.max():.2%}')
print('(High NaN rates expected for sparse vitals/labs — LightGBM handles natively)')

Building feature matrices...
X_train: (11716, 408)  y_train: (11716,)
X_val  : (2554, 408)    y_val  : (2554,)
X_test : (2484, 408)   y_test : (2484,)

NaN rate per feature — mean: 34.12%  max: 91.03%
(High NaN rates expected for sparse vitals/labs — LightGBM handles natively)


In [ ]:
# # Quick check on the existing var_names so you know what's getting renamed.
# from collections import Counter

# dup_counter = Counter(var_names)
# duplicates  = {name: count for name, count in dup_counter.items() if count > 1}

# if duplicates:
#     print(f'Found {len(duplicates)} duplicated variable names:')
#     for name, count in sorted(duplicates.items(), key=lambda x: -x[1]):
#         # Find every column index where this name occurs in the original feature_names
#         positions = [i for i, fn in enumerate(feature_names)
#                      if fn.replace('_val', '') == name and i % 2 == 0]
#         print(f'  {name!r:<30} appears {count}× at val-column indices {positions}')
# else:
#     print('No duplicates found — the error must be elsewhere.')

Found 4 duplicated variable names:
  'sodium'                       appears 2× at val-column indices [42, 62]
  'hemoglobin'                   appears 2× at val-column indices [44, 80]
  'chloride'                     appears 2× at val-column indices [50, 64]
  'anion_gap'                    appears 2× at val-column indices [52, 72]


## 5. Train LightGBM

In [ ]:
import lightgbm as lgb
import time

print(f'Training LightGBM on {len(y_train):,} stays, {num_classes} classes, {n_tab_features} features...')

train_set = lgb.Dataset(X_train, label=y_train, feature_name=tabular_feature_names)
val_set   = lgb.Dataset(X_val,   label=y_val,   feature_name=tabular_feature_names,
                        reference=train_set)

params = {**LGB_PARAMS, 'num_class': num_classes}

start = time.time()
booster = lgb.train(
    params,
    train_set,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[train_set, val_set],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True),
        lgb.log_evaluation(period=50),
    ],
)
elapsed = time.time() - start
print(f'\n✓ Training complete — {elapsed:.1f}s, best iter: {booster.best_iteration}')

Training LightGBM on 11,716 stays, 5 classes, 408 features...
Training until validation scores don't improve for 50 rounds
[50]	train's multi_logloss: 0.425538	val's multi_logloss: 0.755818
[100]	train's multi_logloss: 0.211405	val's multi_logloss: 0.706163
[150]	train's multi_logloss: 0.114019	val's multi_logloss: 0.707862
Early stopping, best iteration is:
[116]	train's multi_logloss: 0.172013	val's multi_logloss: 0.70343

✓ Training complete — 19.6s, best iter: 116


## 6. Evaluate on Test Set

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Predict probabilities on test set
test_probs = booster.predict(X_test, num_iteration=booster.best_iteration)  # (N, num_classes)
test_preds = test_probs.argmax(axis=1)

# Headline metrics
top1_acc    = accuracy_score(y_test, test_preds)
f1_macro    = f1_score(y_test, test_preds, average='macro',    zero_division=0)
f1_weighted = f1_score(y_test, test_preds, average='weighted', zero_division=0)

# Top-K accuracy
topk_acc = {}
for k in [3, 5]:
    topk_preds  = np.argsort(test_probs, axis=1)[:, -k:]
    topk_acc[k] = np.mean([y_test[i] in topk_preds[i] for i in range(len(y_test))])

print('═' * 50)
print('LIGHTGBM TEST SET RESULTS')
print('═' * 50)
print(f'  Top-1 accuracy : {top1_acc*100:6.2f}%')
print(f'  Top-3 accuracy : {topk_acc[3]*100:6.2f}%')
print(f'  F1 macro       : {f1_macro*100:6.2f}%')
print(f'  F1 weighted    : {f1_weighted*100:6.2f}%')
print(f'  Test samples   : {len(y_test):,}')

══════════════════════════════════════════════════
LIGHTGBM TEST SET RESULTS
══════════════════════════════════════════════════
  Top-1 accuracy :  73.87%
  Top-3 accuracy :  96.01%
  F1 macro       :  66.17%
  F1 weighted    :  72.63%
  Test samples   : 2,484


In [ ]:
# Per-class report — useful for the per-class analysis
target_names = [inv_label_map[i] for i in range(num_classes)]
print(classification_report(
    y_test, test_preds,
    target_names=target_names,
    digits=3,
    zero_division=0,
))

                    precision    recall  f1-score   support

            SEPSIS      0.742     0.875     0.803       824
      STROKE_NEURO      0.895     0.876     0.885       724
          ACUTE_MI      0.554     0.335     0.418       367
CORONARY_ARTERY_DZ      0.627     0.756     0.686       320
     HEART_FAILURE      0.587     0.462     0.517       249

          accuracy                          0.739      2484
         macro avg      0.681     0.661     0.662      2484
      weighted avg      0.729     0.739     0.726      2484



In [ ]:
from collections import Counter

# Majority-class baseline: always predict the most common class in training
most_common_label = Counter([c['label'] for c in train_cards]).most_common(1)[0][0]
majority_acc = (y_test == most_common_label).mean()
print(f'Majority-class baseline    : {majority_acc*100:5.2f}%')

# Stratified-random baseline: sample predictions from the training prior
from collections import Counter
prior = np.array([Counter([c['label'] for c in train_cards]).get(i, 0) for i in range(num_classes)])
prior = prior / prior.sum()
np.random.seed(0)
random_preds = np.random.choice(num_classes, size=len(y_test), p=prior)
random_acc = (y_test == random_preds).mean()
print(f'Stratified-random baseline : {random_acc*100:5.2f}%')

# True top-3 ceiling under random: what a random classifier achieves at top-3
# (the prob mass of the 3 most common classes)
top3_classes = np.argsort(prior)[-3:]
random_top3_acc = sum((y_test == c).mean() for c in top3_classes)
print(f'Random top-3 (3 majority)  : {random_top3_acc*100:5.2f}%')

Majority-class baseline    : 33.17%
Stratified-random baseline : 24.68%
Random top-3 (3 majority)  : 75.20%


## 7. Feature Importance

LightGBM gives you free interpretability. The 'gain' importance shows which
summary statistics contribute most to splits — this is also a good sanity
check (heart rate stats and lactate stats should rank high, not e.g. alarm
limit settings).

In [ ]:
import pandas as pd

# Feature importance by gain (impact on loss reduction)
imp = pd.DataFrame({
    'feature'   : tabular_feature_names,
    'gain'      : booster.feature_importance(importance_type='gain'),
    'split_count': booster.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)

print('TOP 30 FEATURES BY GAIN:')
print(imp.head(30).to_string(index=False))

# Save full ranking for the writeup
imp_path = os.path.join(OUTPUT_DIR, 'feature_importance.csv')
imp.to_csv(imp_path, index=False)
print(f'\nFull importance ranking saved → {imp_path}')

TOP 30 FEATURES BY GAIN:
                                    feature         gain  split_count
                       resp_alarm_high__min 12620.507314           78
                      urine_output__missing 12567.682269          328
                            dextrose_5__min  8743.131615          333
                       resp_alarm_high__max  8318.748941           64
                      resp_alarm_high__mean  6387.963472          104
                     spo2_desat_limit__mean  5194.118828          222
                           dextrose_5__mean  4587.289867          348
 non_invasive_blood_pressure_systolic__mean  4536.813757          544
                      resp_alarm_high__last  4276.921107           32
                     spo2_desat_limit__last  4136.962256           90
                             nacl_0_9__mean  3706.441196          424
                      spo2_desat_limit__max  3644.648319          135
                          nacl_0_9__missing  3596.435473         

## 8. Build `debate_inputs.json` (same schema as LSTM notebook)

In [ ]:
# Build the per-test-case structure the agent notebook consumes.
# Schema MUST match the LSTM notebook exactly so swapping classifier requires
# only changing the input path on the agent side.

debate_inputs       = []
lstm_correct_count  = 0   # variable named 'lstm_*' for schema compatibility
true_in_topk_count  = 0

# Re-run prediction one card at a time (slower but matches LSTM notebook structure)
print(f'Building debate inputs for {len(test_cards):,} test cases...')

# Vectorized version — extract_features already gave us X_test, test_probs
top_k_idx_all = np.argsort(test_probs, axis=1)[:, -TOP_K_DEBATE:][:, ::-1]  # descending

for i, card in enumerate(test_cards):
    top_k_idx   = top_k_idx_all[i]
    top_k_codes = [inv_label_map[int(j)] for j in top_k_idx]
    top_k_probs = [float(test_probs[i, int(j)]) for j in top_k_idx]

    true_label = card['label']
    true_code  = inv_label_map[true_label]

    pred_code     = top_k_codes[0]
    is_correct    = (pred_code == true_code)
    is_in_topk    = (true_code in top_k_codes)

    lstm_correct_count += is_correct
    true_in_topk_count += is_in_topk

    debate_inputs.append({
        'patient_id'      : card['patient_id'],
        'visit_id'        : card.get('visit_id', card.get('hadm_id')),
        'stay_id'         : card['stay_id'],
        'true_label'      : true_label,
        'true_icd_code'   : true_code,
        'top_k_codes'     : top_k_codes,
        'top_k_probs'     : top_k_probs,
        'lstm_prediction' : pred_code,        # name kept for agent-side compatibility
        'lstm_confidence' : top_k_probs[0],   # ditto
        'lstm_correct'    : is_correct,       # ditto
        'true_in_topk'    : is_in_topk,
        'clinical_summary': card.get('clinical_summary', {}),
    })

lstm_top1 = lstm_correct_count / len(test_cards)
in_topk   = true_in_topk_count / len(test_cards)

print(f'\nLightGBM Top-1   : {lstm_top1*100:.2f}%')
print(f'True in Top-{TOP_K_DEBATE}     : {in_topk*100:.2f}%  ← agent ceiling')

Building debate inputs for 2,484 test cases...

LightGBM Top-1   : 73.87%
True in Top-3     : 96.01%  ← agent ceiling


## 9. Save Outputs

In [ ]:
import joblib
import pandas as pd

def to_python(obj):
    """Recursively convert numpy types to plain Python for JSON."""
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.bool_,)):    return bool(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    if isinstance(obj, dict):  return {k: to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [to_python(v) for v in obj]
    return obj

# 1. LightGBM booster (the model itself)
model_path = os.path.join(OUTPUT_DIR, 'best_model.txt')   # .txt is LGB native format
booster.save_model(model_path, num_iteration=booster.best_iteration)

# 2. debate_inputs.json — same schema as LSTM notebook
debate_path = os.path.join(OUTPUT_DIR, 'debate_inputs.json')
with open(debate_path, 'w') as f:
    json.dump({
        'classifier_type'  : 'lightgbm',
        'label_map'        : label_map,
        'inv_label_map'    : {str(k): v for k, v in inv_label_map.items()},
        'top_k'            : TOP_K_DEBATE,
        'n_test'           : len(debate_inputs),
        'lstm_top1_acc'    : round(float(lstm_top1), 4),  # name kept for compat
        'true_in_topk_rate': round(float(in_topk),   4),
        'patients'         : to_python(debate_inputs),
    }, f, indent=2)

# 3. test_predictions.csv — flat per-case table for analysis
csv_path = os.path.join(OUTPUT_DIR, 'test_predictions.csv')
pd.DataFrame([{
    'visit_id'    : d['visit_id'],
    'stay_id'     : d['stay_id'],
    'true_icd'    : d['true_icd_code'],
    'pred_icd'    : d['lstm_prediction'],
    'confidence'  : d['lstm_confidence'],
    'correct'     : d['lstm_correct'],
    'true_in_topk': d['true_in_topk'],
    'top_k_codes' : ' | '.join(d['top_k_codes']),
    'top_k_probs' : ' | '.join(f'{p:.3f}' for p in d['top_k_probs']),
} for d in debate_inputs]).to_csv(csv_path, index=False)

# 4. model_config.json
config_path = os.path.join(OUTPUT_DIR, 'model_config.json')
with open(config_path, 'w') as f:
    json.dump({
        'model_type'     : 'LightGBM',
        'lgb_params'     : LGB_PARAMS,
        'num_class'      : num_classes,
        'best_iteration' : int(booster.best_iteration),
        'n_tab_features' : n_tab_features,
        'stat_names'     : STAT_NAMES,
        'var_names'      : var_names,
        'feature_names'  : tabular_feature_names,
        'n_timesteps'    : n_timesteps,
        'label_map'      : label_map,
        'split_seed'     : SPLIT_SEED,
        'train_ratio'    : TRAIN_RATIO,
        'val_ratio'      : VAL_RATIO,
        'split_signature': test_signature,
    }, f, indent=2)

# 5. Persist the test cohort patient_ids (so agent runs can lock to it)
cohort_path = os.path.join(OUTPUT_DIR, 'test_cohort.json')
with open(cohort_path, 'w') as f:
    json.dump({
        'patient_ids' : [c['patient_id'] for c in test_cards],
        'stay_ids'    : [c.get('stay_id') for c in test_cards],
        'signature'   : test_signature,
        'split_seed'  : SPLIT_SEED,
    }, f)

print('Saved outputs:')
for path in [model_path, debate_path, csv_path, config_path, cohort_path,
             os.path.join(OUTPUT_DIR, 'feature_importance.csv')]:
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1e6
        print(f'  {os.path.basename(path):<28} {mb:7.2f} MB')

print(f'\n{"═" * 50}')
print(f'LIGHTGBM BASELINE SUMMARY')
print(f'{"═" * 50}')
print(f'  Top-1 accuracy   : {top1_acc*100:6.2f}%')
print(f'  Top-3 accuracy   : {topk_acc[3]*100:6.2f}%')
print(f'  Agent ceiling    : {in_topk*100:6.2f}%  (true in top-{TOP_K_DEBATE})')
print(f'  Best iteration   : {booster.best_iteration}')

Saved outputs:
  best_model.txt                  4.01 MB
  debate_inputs.json             20.00 MB
  test_predictions.csv            0.32 MB
  model_config.json               0.01 MB
  test_cohort.json                0.06 MB
  feature_importance.csv          0.02 MB

══════════════════════════════════════════════════
LIGHTGBM BASELINE SUMMARY
══════════════════════════════════════════════════
  Top-1 accuracy   :  73.87%
  Top-3 accuracy   :  96.01%
  Agent ceiling    :  96.01%  (true in top-3)
  Best iteration   : 116


## 10. Side-by-Side Comparison with LSTM (optional)

If you've already trained the LSTM, this cell loads its `debate_inputs.json`
and compares per-case agreement to the LightGBM. Useful for the writeup.

In [ ]:
# Set this to the LSTM output dir; if not present, this cell is a no-op
LSTM_OUTPUT_DIR = '/content/drive/MyDrive/702 project/TOP5_lstm_outputs'
lstm_debate = os.path.join(LSTM_OUTPUT_DIR, 'debate_inputs.json')

if not os.path.exists(lstm_debate):
    print(f'(skipped — no LSTM debate_inputs.json at {lstm_debate})')
else:
    with open(lstm_debate) as f:
        lstm_data = json.load(f)

    # Index by stay_id for join
    lstm_by_stay = {p['stay_id']: p for p in lstm_data['patients']}
    lgb_by_stay  = {p['stay_id']: p for p in debate_inputs}

    common_stays = sorted(set(lstm_by_stay) & set(lgb_by_stay))
    print(f'Test stays in both runs: {len(common_stays):,}')
    if len(common_stays) != len(lstm_by_stay) or len(common_stays) != len(lgb_by_stay):
        print(f'  ⚠ LSTM-only stays: {len(set(lstm_by_stay) - set(lgb_by_stay))}')
        print(f'  ⚠ LGB-only  stays: {len(set(lgb_by_stay) - set(lstm_by_stay))}')
        print(f'  → split is not aligned; check seed and cards file')

    # Joint accuracy table
    both_correct  = 0
    only_lstm     = 0
    only_lgb      = 0
    both_wrong    = 0
    same_top1     = 0
    in_topk_lstm  = 0
    in_topk_lgb   = 0
    in_topk_both  = 0

    for sid in common_stays:
        a = lstm_by_stay[sid]
        b = lgb_by_stay[sid]
        if a['lstm_correct'] and b['lstm_correct']:        both_correct += 1
        elif a['lstm_correct'] and not b['lstm_correct']:  only_lstm    += 1
        elif not a['lstm_correct'] and b['lstm_correct']:  only_lgb     += 1
        else:                                              both_wrong   += 1

        if a['lstm_prediction'] == b['lstm_prediction']:   same_top1    += 1
        if a['true_in_topk']:   in_topk_lstm += 1
        if b['true_in_topk']:   in_topk_lgb  += 1
        if a['true_in_topk'] and b['true_in_topk']: in_topk_both += 1

    n = len(common_stays)
    print(f'\nTop-1 accuracy:')
    print(f'  LSTM     : {(both_correct + only_lstm)/n*100:6.2f}%')
    print(f'  LightGBM : {(both_correct + only_lgb)/n*100:6.2f}%')
    print(f'\nAgreement on top-1: {same_top1}/{n} ({same_top1/n*100:.1f}%)')
    print(f'\nCase breakdown (top-1):')
    print(f'  Both correct       : {both_correct} ({both_correct/n*100:.1f}%)')
    print(f'  LSTM only correct  : {only_lstm} ({only_lstm/n*100:.1f}%)')
    print(f'  LGB only correct   : {only_lgb} ({only_lgb/n*100:.1f}%)')
    print(f'  Both wrong         : {both_wrong} ({both_wrong/n*100:.1f}%)')
    print(f'\nTrue in top-{TOP_K_DEBATE} (agent ceiling):')
    print(f'  LSTM     : {in_topk_lstm/n*100:6.2f}%')
    print(f'  LightGBM : {in_topk_lgb/n*100:6.2f}%')
    print(f'  Either   : {(in_topk_lstm + in_topk_lgb - in_topk_both)/n*100:6.2f}%  ← ensemble ceiling')

Test stays in both runs: 2,484

Top-1 accuracy:
  LSTM     :  67.15%
  LightGBM :  73.87%

Agreement on top-1: 1910/2484 (76.9%)

Case breakdown (top-1):
  Both correct       : 1524 (61.4%)
  LSTM only correct  : 144 (5.8%)
  LGB only correct   : 311 (12.5%)
  Both wrong         : 505 (20.3%)

True in top-3 (agent ceiling):
  LSTM     :  93.88%
  LightGBM :  96.01%
  Either   :  97.67%  ← ensemble ceiling
